# 03 · Olive oil production and the official forecast, 2015/16–2026/27

The **aforo** is the Junta de Andalucía's official forecast of the olive oil harvest, published in
October from field fruit counts. The **final production** of each campaign is back-calculated
from the % variation the Junta publishes the next year (ADR 0002; validated against AICA within
0.4 %). Both come from `api/data/production.json`, so this notebook runs anywhere.

Questions: how much does production swing from year to year, and how wrong is the aforo?

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from oliveye.models.plots import GRID, INK, MUTED, SURFACE

production = json.loads(Path("../api/data/production.json").read_text())
table = pd.concat(
    [pd.DataFrame(rows).assign(province=province) for province, rows in production.items()],
    ignore_index=True,
)
table["aforo_error_pct"] = (table["oil_aforo_t"] / table["oil_final_t"] - 1) * 100
# fill_method=None: a missing campaign stays missing instead of becoming a 0 % change.
table["change_pct"] = table.groupby("province")["oil_final_t"].pct_change(fill_method=None) * 100
table.round(1)

## Alternate bearing: production swings by tens of percent

Olive trees alternate heavy and light crops (*vecería*), and droughts (2022/23, 2023/24) cut the
harvest in half. A forecast has to anticipate swings like these, not just the average level.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
fig.patch.set_facecolor(SURFACE)
for ax, (province, rows) in zip(axes, table.groupby("province", sort=False), strict=True):
    rows = rows.dropna(subset=["change_pct"])
    colours = np.where(rows["change_pct"] >= 0, "#054d28", "#cb272f")
    ax.bar(rows["campaign"], rows["change_pct"], color=colours, width=0.65)
    ax.axhline(0, color=INK, linewidth=0.8)
    ax.set_title(province, loc="left", color=INK, fontweight="bold")
    ax.tick_params(axis="x", rotation=60, labelsize=8, colors=MUTED)
    ax.grid(axis="y", color=GRID)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("Change vs previous campaign (%)", color=MUTED)
fig.tight_layout()

## How wrong is the aforo?

Positive = the aforo forecast more oil than was finally produced. It is the bar the OlivEye model
is measured against (MAPE 11.9 % on the comparable rows, `docs/results/`).
Most "final" values are derived from the Junta's later % variations (ADR 0002), not measured.

In [ ]:
errors = table.dropna(subset=["aforo_error_pct"])
fig, ax = plt.subplots(figsize=(10, 3.6))
fig.patch.set_facecolor(SURFACE)
width = 0.38
campaigns = sorted(errors["campaign"].unique())
x = np.arange(len(campaigns))
for offset, (province, colour) in zip(
    (-width / 2, width / 2), (("Jaén", "#054d28"), ("Granada", "#2a78d6")), strict=True
):
    rows = errors[errors["province"] == province].set_index("campaign").reindex(campaigns)
    ax.bar(x + offset, rows["aforo_error_pct"], width=width, color=colour, label=province)
ax.axhline(0, color=INK, linewidth=0.8)
ax.set_xticks(x, campaigns, rotation=45, ha="right")
ax.set_ylabel("Aforo error vs derived final (%)", color=MUTED)
ax.grid(axis="y", color=GRID)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
fig.tight_layout()
errors.groupby("province")["aforo_error_pct"].agg(
    mean_abs=lambda s: s.abs().mean(), worst="max", best="min"
).round(1)

**Take-aways.** Year-to-year swings run from −64 % to +172 % (Jaén 2022/23 drought, 2024/25 recovery), and even the aforo, which counts
fruit in the field, misses by up to ~20–30 % in a bad year (Jaén 2020/21: +28 %). With only 9
comparable campaigns, beating it from space with data up to August is a hard target — which is
exactly what the results show.